# Medical Text Interpretation using Ollama Med42

Analyzing medical text explanations using interpret-text with Ollama's med42 model.

In [2]:
!pip install interpret-text

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached setuptools-78.1.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-14.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached jmespath-1.0.1-py3-none-any.whl.metadata (7.6 kB)
  Using cached markdown_it_py-3.0.0-py3-none-any.whl.metadata (6.9 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.0/22.0 MB 4.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 4.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 4.9 MB/s eta 0:00:00a 0:00:01
Using cached annotate

In [ ]:
import sys
sys.path.append("..")
import os
import json
import requests
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from interpret_text.experimental.introspective_rationale import IntrospectiveRationaleExplainer
from utils.medical_processor import MedicalTermProcessor
from lime.lime_text import LimeTextExplainer
import matplotlib.pyplot as plt

In [4]:
# Configuration
QUICK_RUN = True  # For testing
MODEL_TYPE = "RNN"
CUDA = torch.cuda.is_available()

# Medical text processing parameters
DATA_FOLDER = "../data/medical"
LABEL_COL = "relevance"  # Binary classification: medical relevance
TEXT_COL = "text"
MAX_TOKENS = 100
TOKEN_THRESHOLD = 1

# Model configuration
model_config = {
    "cuda": CUDA,
    "model_save_dir": "../models",
    "model_prefix": "medical_introspective",
    "lr": 2e-4,
    "batch_size": 32
}

if QUICK_RUN:
    model_config.update({
        "save_best_model": False,
        "pretrain_cls": True,
        "num_epochs": 1
    })

print("Configuration initialized")

Configuration initialized


In [5]:
# Create sample medical dataset
def create_medical_dataset():
    medical_processor = MedicalTermProcessor()
    
    # Sample medical texts
    texts = [
        "Patient presents with severe headache and fever",
        "Blood pressure is elevated at 160/95",
        "The weather is nice today",  # Non-medical
        "Chest x-ray shows bilateral infiltrates",
        "I like watching movies",  # Non-medical
        "Patient reports chronic joint pain"
    ]
    
    # Determine medical relevance
    relevance = []
    for text in texts:
        words = set(text.lower().split())
        is_medical = bool(words & medical_processor.medical_terms)
        relevance.append(1 if is_medical else 0)
    
    return pd.DataFrame({
        TEXT_COL: texts,
        LABEL_COL: relevance
    })

# Create and split dataset
data = create_medical_dataset()
train_size = int(0.8 * len(data))
train_data = data[:train_size]
test_data = data[train_size:]

print(f"Created dataset with {len(data)} samples")
print("\nSample data:")
print(data.head())

Created dataset with 6 samples

Sample data:
                                              text  relevance
0  Patient presents with severe headache and fever          1
1             Blood pressure is elevated at 160/95          0
2                        The weather is nice today          0
3          Chest x-ray shows bilateral infiltrates          0
4                           I like watching movies          0


In [6]:
# Initialize preprocessor and prepare data
preprocessor = GlovePreprocessor(TOKEN_THRESHOLD, MAX_TOKENS)
preprocessor.build_vocab(data[TEXT_COL])

# Preprocess data
df_train = pd.concat([train_data[LABEL_COL], preprocessor.preprocess(train_data[TEXT_COL])], axis=1)
df_test = pd.concat([test_data[LABEL_COL], preprocessor.preprocess(test_data[TEXT_COL])], axis=1)

# Update model config
model_config["labels"] = np.array([0, 1])  # Binary classification
model_config["num_labels"] = 2

print("Data preprocessing complete")

Data preprocessing complete


In [ ]:
# Ollama configuration
OLLAMA_CONFIG = {
    "url": "http://localhost:11434/api/chat",
    "model": "llama3-med42-8b",
    "system_prompt": "You are a medical expert analyzing clinical information."
}

def get_ollama_response(text):
    """Get response from Ollama model"""
    try:
        payload = {
            "model": OLLAMA_CONFIG["model"],
            "messages": [
                {"role": "system", "content": OLLAMA_CONFIG["system_prompt"]},
                {"role": "user", "content": text}
            ]
        }
        response = requests.post(OLLAMA_CONFIG["url"], json=payload, timeout=30)
        if response.status_code == 200:
            return response.json()["message"]["content"]
        return f"Error: {response.status_code}"
    except Exception as e:
        return f"Error: {str(e)}"

# Initialize processors
medical_processor = MedicalTermProcessor()
lime_explainer = LimeTextExplainer(
    class_names=['not_relevant', 'relevant'],
    split_expression='\s+',
    random_state=42
)

# Test model connection
test_response = get_ollama_response("Test connection")
print("Model initialized and tested.")

In [ ]:
def predictor_fn(texts):
    """Prediction function for LIME"""
    predictions = []
    print(f"Processing {len(texts)} samples...")
    
    for text in texts:
        # Get model response
        response = get_ollama_response(text)
        
        # Calculate medical relevance
        text_terms = set(text.lower().split())
        resp_terms = set(response.lower().split())
        medical_terms = medical_processor.medical_terms
        
        # Check medical term overlap
        medical_overlap = len((text_terms | resp_terms) & medical_terms)
        relevance = min(0.5 + (medical_overlap * 0.1), 0.9)
        
        predictions.append([1 - relevance, relevance])
    
    return np.array(predictions)

# Test predictor
test_pred = predictor_fn(["Patient has severe headache"])
print("\nTest prediction:", test_pred)

In [ ]:
def analyze_medical_text(text):
    """Analyze medical text using Ollama and LIME"""
    print(f"\nAnalyzing: {text}")
    
    # Get model response
    response = get_ollama_response(text)
    print(f"Model response: {response}")
    
    # Generate LIME explanation
    exp = lime_explainer.explain_instance(
        text,
        predictor_fn,
        num_features=10,
        num_samples=100
    )
    
    # Get feature importance
    feature_importance = exp.as_list()
    
    # Visualize
    plt.figure(figsize=(12, 6))
    features, scores = zip(*feature_importance)
    y_pos = range(len(features))
    
    plt.barh(y_pos, scores, 
            color=['red' if s > 0 else 'blue' for s in scores])
    plt.yticks(y_pos, features)
    plt.xlabel('Impact on Medical Relevance')
    plt.title('Feature Importance Analysis')
    plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
    plt.grid(True, axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    return pd.DataFrame(feature_importance, columns=['Feature', 'Importance'])

In [ ]:
# Test cases
test_cases = [
    "Patient presents with acute abdominal pain and nausea",
    "The ECG shows ST elevation in leads V1-V4",
    "Blood glucose levels are consistently above 200 mg/dL"
]

# Analyze first test case
importance_df = analyze_medical_text(test_cases[0])
print("\nFeature Importance Summary:")
print(importance_df.sort_values('Importance', ascending=False))

In [ ]:
# Test explanation stability
def test_stability(text, runs=3):
    """Test explanation stability across multiple runs"""
    all_results = []
    
    for i in range(runs):
        print(f"\nRun {i+1}/{runs}")
        results = analyze_medical_text(text)
        all_results.append(results)
    
    return all_results

# Test stability with second case
stability_results = test_stability(test_cases[1])